# Predictive Customer Churn Classifier
## Random Forest and XGBoost

**Dataset:** `Telco-Customer-Churn.csv`  
**Goal:** Predict whether a telecom customer will leave (`Churn = Yes`) and compare Random Forest with XGBoost using ROC-AUC.

### Workflow
1. Load and inspect the dataset.
2. Clean `TotalCharges` and remove the customer ID.
3. Split the data using a stratified train/test split.
4. One-hot encode categorical features.
5. Address class imbalance with class weights (`class_weight` for Random Forest and `scale_pos_weight` for XGBoost).
6. Evaluate with 5-fold cross-validation, ROC-AUC, ROC curves, confusion matrices, and feature importance.

> This notebook uses class weighting instead of SMOTE. All preprocessing is placed inside a scikit-learn pipeline to reduce data leakage. Results may vary slightly by library version and split.

## 1. Install required packages

Run this cell once in your notebook environment. If XGBoost is already installed, you can skip its installation.

In [ ]:
%pip install -U pandas numpy matplotlib seaborn scikit-learn xgboost

## 2. Import libraries and load the dataset

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from xgboost import XGBClassifier

RANDOM_STATE = 42

# Keep Telco-Customer-Churn.csv in the same folder as this notebook.
df = pd.read_csv("Telco-Customer-Churn.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())

## 3. Inspect and clean the data

In [ ]:
print("Data types and non-null counts:")
df.info()

print("\nChurn distribution:")
print(df["Churn"].value_counts())
print("\nChurn percentages:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

# TotalCharges is stored as text in this dataset. Blank strings become NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove the ID column because it identifies a customer but is not a useful
# generalizable predictor of churn.
df = df.drop(columns=["customerID"], errors="ignore")

print("\nMissing values after converting TotalCharges:")
print(df.isna().sum().sort_values(ascending=False).head())
print("\nUpdated shape:", df.shape)

## 4. Separate features and target

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

if y.isna().any():
    raise ValueError("Unexpected target labels found. Check the values in the Churn column.")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)
print("\nTarget counts (0 = No churn, 1 = Churn):")
print(y.value_counts().sort_index())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("\nTraining rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 5. Build preprocessing and models

- Numeric columns: median imputation and standardization.
- Categorical columns: most-frequent imputation and one-hot encoding.
- Random Forest: uses `class_weight="balanced"`.
- XGBoost: uses `scale_pos_weight` based on the training split.

One-hot encoding is fitted only on training data through the pipeline.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Class imbalance ratio calculated only from training data.
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

print(f"XGBoost scale_pos_weight: {scale_pos_weight:.3f}")

## 6. Evaluate with 5-fold stratified cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1
    )
    cv_results.append({
        "Model": name,
        "Mean CV ROC-AUC": scores.mean(),
        "Std CV ROC-AUC": scores.std()
    })
    print(f"{name}: fold ROC-AUC = {np.round(scores, 4)}")
    print(f"Mean ROC-AUC = {scores.mean():.4f} ± {scores.std():.4f}\n")

cv_results_df = pd.DataFrame(cv_results).sort_values(
    "Mean CV ROC-AUC", ascending=False
)
display(cv_results_df)

## 7. Train models and evaluate on the held-out test set

In [ ]:
test_results = []
fitted_models = {}
test_probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    test_probabilities[name] = y_prob

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }
    test_results.append(metrics)

    print("=" * 60)
    print(name)
    print("=" * 60)
    print(classification_report(
        y_test, y_pred, target_names=["No Churn", "Churn"], zero_division=0
    ))

results_df = pd.DataFrame(test_results).sort_values("ROC-AUC", ascending=False)
display(results_df.style.format({
    "Accuracy": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}",
    "F1-score": "{:.3f}", "ROC-AUC": "{:.3f}"
}))

## 8. Plot ROC-AUC curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, y_prob in test_probabilities.items():
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=name, ax=plt.gca()
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.title("ROC Curves — Customer Churn Prediction")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Plot confusion matrices

In [ ]:
for name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["No Churn", "Churn"],
        cmap="Blues",
        values_format="d"
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

## 10. Display the top 10 influential features

The feature names are taken from the fitted one-hot encoder, so categorical values appear as separate encoded features. These importances describe model reliance; they do not prove causation.

In [ ]:
def get_feature_names(fitted_pipeline):
    fitted_preprocessor = fitted_pipeline.named_steps["preprocessor"]
    return fitted_preprocessor.get_feature_names_out()

for name, model in fitted_models.items():
    classifier = model.named_steps["classifier"]
    feature_names = get_feature_names(model)
    importances = classifier.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False).head(10)

    print(f"Top 10 features — {name}")
    display(importance_df)

    plt.figure(figsize=(9, 6))
    sns.barplot(data=importance_df, x="Importance", y="Feature")
    plt.title(f"Top 10 Feature Importances — {name}")
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

## 11. Optional: Predict churn for a new customer

Enter values using the original dataset's column names. The example values below are illustrative only; change them to match a real record. The model returns a probability, not a guaranteed outcome.

In [ ]:
# Example customer record (edit values before running)
new_customer = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 70.0,
    "TotalCharges": 840.0
}])

# Choose the model with the highest held-out test ROC-AUC.
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

churn_probability = best_model.predict_proba(new_customer)[0, 1]
predicted_class = "Yes — likely to churn" if churn_probability >= 0.5 else "No — unlikely to churn"

print("Selected model:", best_model_name)
print("Predicted churn class:", predicted_class)
print(f"Estimated churn probability: {churn_probability:.2%}")

## 12. Conclusion template

After running all cells, use the actual values printed by your notebook to complete this section:

- **Best model by test ROC-AUC:** [fill from `results_df`]
- **Test ROC-AUC:** [copy the measured value]
- **Precision / Recall / F1-score:** [copy measured values]
- **Important features:** [report the top features shown by the importance plots]

Do not claim that ROC-AUC is above 0.80 unless your actual results confirm it. If performance is lower than expected, tune model parameters and evaluate on the same held-out test set only after the tuning process is finalized.